# LEPTOLIC — Demo ทดลองกรอกค่า

ฟอร์มสำหรับโมเดลอ้างอิง **befec00** (Logistic Regression C=0.1 · อายุ เพศ SBP DBP น้ำหนัก สูบบุหรี่)

**วิธีใช้:** `Runtime → Run all` แล้วเลื่อนลงไปที่ฟอร์ม กรอกค่า กด **ทำนาย**

ผลคือ *คะแนนทดลอง* ของโอกาสที่จะ **รายงานว่าได้รับการวินิจฉัยความดันสูงครั้งแรกภายใน 8 ปี**
จากข้อมูล NHEFS สหรัฐฯ ปี 1982–1992 — ไม่ใช่การวินิจฉัย และยังไม่ได้ทดสอบกับคนไทย

In [1]:
# ถ้าเปิดใน Google Colab จะ clone repo มาให้อัตโนมัติ (รันบนเครื่องตัวเองข้ามได้)
from pathlib import Path
import os, subprocess
BRANCH = 'dev-by-gunner'
if not Path('model/model.json').exists():
    if not Path('Lebtolic').exists():
        subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                        'https://github.com/s4arry/Lebtolic.git'], check=True)
    os.chdir('Lebtolic')
print('working directory:', Path.cwd())

working directory: /home/claude/repo/Lebtolic


In [2]:
from demo.form import Answers, InputRejected, PRESETS, SMOKING_TH, FEATURE_TH, load_model, predict

model, state = load_model()
THRESHOLD = state['threshold_policies']['fpr145']['threshold']
print(f'โหลดโมเดลแล้ว · {len(state["features"])} encoded columns · threshold {THRESHOLD:.2%}')

โหลดโมเดลแล้ว · 9 encoded columns · threshold 9.80%


## ฟอร์ม

- **ปุ่มตัวอย่าง** ต่ำ / กลาง / สูง เติมค่าสมมติให้ (ไม่ใช่คนจริง)
- **Guardrails:** ระบบจะปฏิเสธถ้าเคยวินิจฉัยแล้ว, ความดัน ≥ 130/80, หรือค่าอยู่นอกช่วงข้อมูลที่โมเดลเรียน

In [3]:
import ipywidgets as W
from IPython.display import display, HTML

style = {'description_width': '150px'}
age = W.FloatText(value=45, description='อายุ (ปี)', style=style)
sex = W.ToggleButtons(options=[('ชาย', 'male'), ('หญิง', 'female')], value='female', description='เพศ', style=style)
sbp = W.FloatText(value=113, description='ความดันตัวบน', style=style)
dbp = W.FloatText(value=71, description='ความดันตัวล่าง', style=style)
weight = W.FloatText(value=67, description='น้ำหนัก', style=style)
unit = W.ToggleButtons(options=['kg', 'lb'], value='kg', style={'button_width': '50px'})
no_weight = W.Checkbox(value=False, description='ไม่ทราบน้ำหนัก', indent=False)
smoking = W.Dropdown(options=[(v, k) for k, v in SMOKING_TH.items()], value='never', description='การสูบบุหรี่', style=style)
diagnosed = W.Checkbox(value=False, description='เคยได้รับการวินิจฉัยว่าเป็นความดันสูงแล้ว', indent=False)
go = W.Button(description='ทำนาย', button_style='primary', icon='heartbeat')
presets = [W.Button(description=f'ตัวอย่าง: {k}') for k in PRESETS]
out = W.Output()

def fill(name):
    a = PRESETS[name]
    age.value, sex.value, sbp.value, dbp.value = a.age, a.sex, a.sbp, a.dbp
    weight.value, unit.value, no_weight.value = a.weight, a.weight_unit, False
    smoking.value, diagnosed.value = a.smoking, False
    run()

def bar(score):
    pct, cut = min(score, 0.5) / 0.5 * 100, THRESHOLD / 0.5 * 100
    color = '#c0392b' if score >= THRESHOLD else '#16a085'
    return f'''<div style="position:relative;height:18px;background:#eee;border-radius:9px;width:420px;margin:6px 0">
      <div style="width:{pct:.1f}%;height:100%;background:{color};border-radius:9px"></div>
      <div style="position:absolute;left:{cut:.1f}%;top:-4px;height:26px;border-left:2px dashed #333"></div></div>
      <div style="width:420px;font-size:12px;color:#666">0% ············ เส้นประ = threshold {THRESHOLD:.1%} ············ 50%+</div>'''

def run(_=None):
    out.clear_output()
    answers = Answers(age=age.value, sex=sex.value, sbp=sbp.value, dbp=dbp.value,
                      weight=None if no_weight.value else weight.value, weight_unit=unit.value,
                      smoking=smoking.value, previously_diagnosed=diagnosed.value)
    with out:
        try:
            r = predict(answers, model, state)
        except InputRejected as e:
            display(HTML(f'<div style="padding:10px;border-left:5px solid #e67e22;background:#fdf2e9">'
                         f'<b>ระบบไม่ทำนาย</b><br>{e}</div>'))
            return
        verdict = ('<b style="color:#c0392b">เกิน threshold → แจ้งเตือน</b>' if r.alert
                   else '<b style="color:#16a085">ต่ำกว่า threshold → ไม่แจ้งเตือน</b>')
        rows = ''.join(f'<tr><td>{FEATURE_TH[k]}</td><td style="text-align:right">{v:+.2f}</td></tr>'
                       for k, v in sorted(r.contributions.items(), key=lambda kv: -abs(kv[1])) if abs(v) >= 0.005)
        warn = ''.join(f'<div style="color:#b9770e">⚠ {w}</div>' for w in r.warnings)
        display(HTML(f'''<div style="font-size:15px">คะแนนทดลอง 8 ปี: <b style="font-size:22px">{r.score:.1%}</b>
          &nbsp;(ค่าเฉลี่ยของกลุ่มข้อมูล 6.1%)</div>{bar(r.score)}<div>{verdict}</div>{warn}
          <details style="margin-top:8px"><summary>ทำไมได้คะแนนนี้ (log-odds เทียบคนเฉลี่ยในข้อมูล)</summary>
          <table>{rows}</table><div style="font-size:12px;color:#666">บวก = ดันคะแนนขึ้น · ลบ = ดึงลง</div></details>
          <div style="font-size:12px;color:#888;margin-top:8px">ข้อมูล NHEFS สหรัฐฯ 1982–1992 · label คือการรายงานการวินิจฉัย ·
          ไม่ใช่การวินิจฉัยหรือคำแนะนำทางการแพทย์</div>'''))

go.on_click(run)
for b, name in zip(presets, PRESETS):
    b.on_click(lambda _, n=name: fill(n))

display(W.VBox([W.HBox(presets), age, sex, sbp, dbp, W.HBox([weight, unit, no_weight]),
                smoking, diagnosed, go, out]))

## ทดสอบโดยไม่ใช้ฟอร์ม (เผื่อ widget ไม่แสดง)

เซลล์นี้รันตัวอย่าง 3 ระดับ และกรณีที่ระบบต้องปฏิเสธ — ใช้ตอน demo หัวข้อ Guardrails ได้

In [4]:
for name, a in PRESETS.items():
    r = predict(a, model, state)
    print(f'{name:>5}: {r.score:6.2%}  แจ้งเตือน={r.alert}')

print()
cases = {
    'เคยวินิจฉัยแล้ว': Answers(50, 'male', 120, 75, 70, previously_diagnosed=True),
    'ความดัน 135/85': Answers(50, 'male', 135, 85, 70),
    'อายุ 20 ปี': Answers(20, 'male', 115, 70, 70),
    'น้ำหนัก 200 kg': Answers(50, 'male', 115, 70, 200),
    'กรอกความดันสลับ': Answers(50, 'male', 70, 115, 70),
}
for name, a in cases.items():
    try:
        predict(a, model, state); print(f'{name}: ผ่าน (ไม่ควรเกิด)')
    except InputRejected as e:
        print(f'{name}: ปฏิเสธ → {e}')

  ต่ำ:  0.80%  แจ้งเตือน=False
 กลาง:  6.93%  แจ้งเตือน=False
  สูง: 42.01%  แจ้งเตือน=True

เคยวินิจฉัยแล้ว: ปฏิเสธ → โมเดลนี้ใช้กับคนที่ยังไม่เคยได้รับการวินิจฉัยความดันสูงเท่านั้น
ความดัน 135/85: ปฏิเสธ → ความดัน 135/85 สูงเกินเกณฑ์ของกลุ่มข้อมูล (ต้องต่ำกว่า 130/80) — โมเดลทำนายเฉพาะคนที่ความดันยังปกติ
อายุ 20 ปี: ปฏิเสธ → อายุ 20 ปี อยู่นอกช่วงข้อมูลที่โมเดลเรียน (32–85 ปี)
น้ำหนัก 200 kg: ปฏิเสธ → น้ำหนัก 200 kg อยู่นอกช่วงที่ข้อมูลรองรับ (31–136 kg หรือ 69–300 lb)
กรอกความดันสลับ: ปฏิเสธ → ความดันตัวล่างต้องน้อยกว่าตัวบน — ตรวจว่ากรอกสลับกันหรือไม่
